# Step 5 Part B: LSTM model + differentiable P&L simulation

The model reads the episode's feature sequence and outputs a hedge position at each step. We then run the SAME P&L logic as the classical benchmarks, but using PyTorch tensor operations so gradients can flow back through the whole simulation -- that's what lets us train the network to directly minimize hedging risk, not just imitate BS delta.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data = np.load("train_episode_tensors.npz")
features = torch.tensor(data["features"], dtype=torch.float32)
spots = torch.tensor(data["spots"], dtype=torch.float32)
option_pnls = torch.tensor(data["option_pnls"], dtype=torch.float32)
masks = torch.tensor(data["masks"], dtype=torch.float32)

print(f"features: {features.shape}, spots: {spots.shape}, option_pnls: {option_pnls.shape}, masks: {masks.shape}")

BTC_TRANSACTION_COST_RATE = 0.0005

## The LSTM policy network

In [ ]:
class DeepHedgePolicy(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)          # (batch, seq_len, hidden_size)
        raw_position = self.head(lstm_out)  # (batch, seq_len, 1)
        # tanh bounds output to [-1.5, 1.5] -- a bit wider than BS delta's [-1,1] range,
        # giving the network some room to be more aggressive if it learns that's useful
        position = 1.5 * torch.tanh(raw_position.squeeze(-1))
        return position  # (batch, seq_len)

model = DeepHedgePolicy().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params}")

## Differentiable P&L simulation

Same formula as the classical engine: trade = position.diff() (first step trade = position[0] - 0), cost = |trade| * spot * rate/2, hedge_pnl = position_lagged * spot.diff(), total_pnl = option_pnl + hedge_pnl - cost. All done with tensor ops across the whole batch at once, respecting the mask so padded steps contribute nothing.

In [ ]:
def simulate_pnl_batch(positions, spots, option_pnls, masks, cost_rate=BTC_TRANSACTION_COST_RATE):
    """
    positions, spots, option_pnls, masks: (batch, seq_len) tensors
    Returns: terminal_pnl per episode (batch,), and per-step total_pnl (batch, seq_len) for diagnostics
    """
    batch_size, seq_len = positions.shape

    # trade[t] = position[t] - position[t-1], with position[-1] treated as 0
    prev_position = torch.cat([torch.zeros(batch_size, 1, device=positions.device), positions[:, :-1]], dim=1)
    trade = positions - prev_position
    trade = trade * masks  # zero out trades on padded steps

    cost = trade.abs() * spots * (cost_rate / 2)

    prev_spot = torch.cat([spots[:, :1], spots[:, :-1]], dim=1)
    spot_diff = spots - prev_spot
    hedge_pnl = prev_position * spot_diff
    # first step has no prior position, so hedge_pnl there is 0 (prev_position is already 0 there)

    total_pnl_per_step = (option_pnls + hedge_pnl - cost) * masks
    terminal_pnl = total_pnl_per_step.sum(dim=1)  # (batch,)

    return terminal_pnl, total_pnl_per_step

# Quick sanity check: run the UNTRAINED model on a small batch
sample_features = features[:8].to(device)
sample_spots = spots[:8].to(device)
sample_option_pnls = option_pnls[:8].to(device)
sample_masks = masks[:8].to(device)

with torch.no_grad():
    sample_positions = model(sample_features)
    sample_terminal_pnl, _ = simulate_pnl_batch(sample_positions, sample_spots, sample_option_pnls, sample_masks)

print("Untrained model -- positions for first episode (first 10 steps):")
print(sample_positions[0, :10].cpu().numpy())
print(f"\nTerminal P&L for first 8 episodes (should be some random-ish values, not NaN/inf):")
print(sample_terminal_pnl.cpu().numpy())

## CVaR loss function

We minimize the CVaR (expected shortfall) of LOSSES at the 95% level -- i.e. we care most about the worst 5% of outcomes in each training batch, not just the average. This directly targets tail risk, matching the proposal's committed risk objective.

In [ ]:
def cvar_loss(terminal_pnl, alpha=0.95):
    """
    terminal_pnl: (batch,) -- positive is profit, negative is loss.
    We want to MINIMIZE the expected value of the worst (1-alpha) fraction of LOSSES.
    Losses = -terminal_pnl. CVaR_alpha(losses) = average of the worst (1-alpha) losses.
    """
    losses = -terminal_pnl
    k = max(1, int((1 - alpha) * losses.shape[0]))
    worst_losses, _ = torch.topk(losses, k)
    return worst_losses.mean()

test_loss = cvar_loss(sample_terminal_pnl)
print(f"CVaR loss on the sample (untrained) batch: {test_loss.item():.4f}")
print("(This should be a positive number -- the average of the worst losses in the batch)")